In [0]:
import json
import os

# Config path inside Databricks Repo
config_path = "../config/dev_config.json"

with open(config_path, "r") as file:
    config = json.load(file)

print(config)

In [0]:
catalog_name = config["catalog"]

source_base_path = config["source"]["base_path"]
source_files = config["source"]["files"]

raw_schema = config["raw"]["schema"]
refined_schema = config["refined"]["schema"]
enriched_schema = config["enriched"]["schema"]
aggregate_schema = config["aggregate"]["schema"]

raw_tables = config["raw"]["tables"]
enriched_tables = config["enriched"]["tables"]
aggregate_tables = config["aggregate"]["tables"]

print("Catalog:", catalog_name)
print("Source base path:", source_base_path)
print("Raw schema:", raw_schema)
print("Refined schema:", refined_schema)
print("Enriched schema:", enriched_schema)
print("Aggregate schema:", aggregate_schema)

In [0]:
spark.sql(f"USE CATALOG {catalog_name}")

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog_name}.{raw_schema}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog_name}.{refined_schema}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog_name}.{enriched_schema}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog_name}.{aggregate_schema}")

print("Schemas created or already exist.")

In [0]:
catalog_name = config["catalog"]

raw_schema = config["raw"]["schema"]
refined_schema = config["refined"]["schema"]
enriched_schema = config["enriched"]["schema"]
aggregate_schema = config["aggregate"]["schema"]

raw_tables = config["raw"]["tables"]
refined_tables = config["refined"]["tables"]
enriched_tables = config["enriched"]["tables"]
aggregate_tables = config["aggregate"]["tables"]

raw_orders_table = f"`{catalog_name}`.`{raw_schema}`.`{raw_tables['orders']}`"
raw_customers_table = f"`{catalog_name}`.`{raw_schema}`.`{raw_tables['customers']}`"
raw_products_table = f"`{catalog_name}`.`{raw_schema}`.`{raw_tables['products']}`"

refined_orders_table = f"`{catalog_name}`.`{refined_schema}`.`{refined_tables['orders']}`"
refined_customers_table = f"`{catalog_name}`.`{refined_schema}`.`{refined_tables['customers']}`"
refined_products_table = f"`{catalog_name}`.`{refined_schema}`.`{refined_tables['products']}`"

enriched_customers_table = f"`{catalog_name}`.`{enriched_schema}`.`{enriched_tables['customers']}`"
enriched_products_table = f"`{catalog_name}`.`{enriched_schema}`.`{enriched_tables['products']}`"
enriched_orders_table = f"`{catalog_name}`.`{enriched_schema}`.`{enriched_tables['orders']}`"

aggregate_profit_table = f"`{catalog_name}`.`{aggregate_schema}`.`{aggregate_tables['profit_by_year_category_subcategory_customer']}`"

In [0]:
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {raw_orders_table} (
    row_id STRING,
    order_id STRING,
    order_date STRING,
    ship_date STRING,
    ship_mode STRING,
    customer_id STRING,
    product_id STRING,
    quantity STRING,
    price STRING,
    discount STRING,
    profit STRING,
    source_file_name STRING,
    ingestion_timestamp TIMESTAMP
)
USING DELTA
CLUSTER BY AUTO
TBLPROPERTIES (
    'delta.enableChangeDataFeed' = 'true'
)
""")

spark.sql(f"""
CREATE TABLE IF NOT EXISTS {raw_customers_table} (
    customer_id STRING,
    customer_name STRING,
    email STRING,
    phone STRING,
    address STRING,
    segment STRING,
    country STRING,
    city STRING,
    state STRING,
    postal_code STRING,
    region STRING,
    source_file_name STRING,
    ingestion_timestamp TIMESTAMP
)
USING DELTA
CLUSTER BY AUTO
TBLPROPERTIES (
    'delta.enableChangeDataFeed' = 'true'
)
""")

spark.sql(f"""
CREATE TABLE IF NOT EXISTS {raw_products_table} (
    product_id STRING,
    category STRING,
    sub_category STRING,
    product_name STRING,
    state STRING,
    price_per_product STRING,
    source_file_name STRING,
    ingestion_timestamp TIMESTAMP
)
USING DELTA
CLUSTER BY AUTO
TBLPROPERTIES (
    'delta.enableChangeDataFeed' = 'true'
)
""")

print("Raw tables created or already exist.")

In [0]:
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {refined_orders_table} (
    row_id BIGINT,
    order_id STRING,
    order_date DATE,
    ship_date DATE,
    ship_mode STRING,
    customer_id STRING,
    product_id STRING,
    quantity INT,
    price DOUBLE,
    discount DOUBLE,
    profit DOUBLE,
    ingestion_timestamp TIMESTAMP
)
USING DELTA
CLUSTER BY AUTO
TBLPROPERTIES (
    'delta.enableChangeDataFeed' = 'true'
)
""")

spark.sql(f"""
CREATE TABLE IF NOT EXISTS {refined_customers_table} (
    customer_id STRING,
    customer_name STRING,
    email STRING,
    phone STRING,
    address STRING,
    segment STRING,
    country STRING,
    city STRING,
    state STRING,
    postal_code STRING,
    region STRING,
    ingestion_timestamp TIMESTAMP
)
USING DELTA
CLUSTER BY AUTO
TBLPROPERTIES (
    'delta.enableChangeDataFeed' = 'true'
)
""")

spark.sql(f"""
CREATE TABLE IF NOT EXISTS {refined_products_table} (
    product_id STRING,
    category STRING,
    sub_category STRING,
    product_name STRING,
    state STRING,
    price_per_product DOUBLE,
    ingestion_timestamp TIMESTAMP
)
USING DELTA
CLUSTER BY AUTO
TBLPROPERTIES (
    'delta.enableChangeDataFeed' = 'true'
)
""")

print("Refined tables created or already exist.")

In [0]:
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {enriched_customers_table} (
    customer_id STRING,
    customer_name STRING,
    email STRING,
    phone STRING,
    address STRING,
    segment STRING,
    country STRING,
    city STRING,
    state STRING,
    postal_code STRING,
    region STRING,
    ingestion_timestamp TIMESTAMP
)
USING DELTA
CLUSTER BY AUTO
""")

spark.sql(f"""
CREATE TABLE IF NOT EXISTS {enriched_products_table} (
    product_id STRING,
    product_name STRING,
    category STRING,
    sub_category STRING,
    state STRING,
    price_per_product DOUBLE,
    ingestion_timestamp TIMESTAMP
)
USING DELTA
CLUSTER BY AUTO
""")

spark.sql(f"""
CREATE TABLE IF NOT EXISTS {enriched_orders_table} (
    row_id BIGINT,
    order_id STRING,
    order_date DATE,
    ship_date DATE,
    ship_mode STRING,
    order_year INT,
    customer_id STRING,
    customer_name STRING,
    country STRING,
    product_id STRING,
    product_name STRING,
    product_category STRING,
    product_sub_category STRING,
    quantity INT,
    price DOUBLE,
    discount DOUBLE,
    profit DECIMAL(18,2),
    ingestion_timestamp TIMESTAMP
)
USING DELTA
CLUSTER BY AUTO
""")

print("Enriched tables created or already exist.")

In [0]:
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {aggregate_profit_table} (
    order_year INT,
    product_category STRING,
    product_sub_category STRING,
    customer_id STRING,
    customer_name STRING,
    total_profit DECIMAL(18,2)
)
USING DELTA
CLUSTER BY AUTO
""")

In [0]:
orders_path = f"{source_base_path}/{source_files['orders']['file_name']}"
customers_path = f"{source_base_path}/{source_files['customers']['file_name']}"
products_path = f"{source_base_path}/{source_files['products']['file_name']}"

print("Orders path   :", orders_path)
print("Customers path:", customers_path)
print("Products path :", products_path)

In [0]:
def file_exists(file_path: str) -> bool:
    try:
        dbutils.fs.ls(file_path)
        return True
    except Exception:
        return False


source_file_paths = {
    "orders": orders_path,
    "customers": customers_path,
    "products": products_path
}

missing_files = []

for file_name, file_path in source_file_paths.items():
    if file_exists(file_path):
        print(f"Found {file_name}: {file_path}")
    else:
        print(f"Missing {file_name}: {file_path}")
        missing_files.append(file_path)

if missing_files:
    raise FileNotFoundError(f"Missing source files: {missing_files}")

print("All source files are available.")

In [0]:
target_tables = {
    "raw_orders": f"{catalog_name}.{raw_schema}.{raw_tables['orders']}",
    "raw_customers": f"{catalog_name}.{raw_schema}.{raw_tables['customers']}",
    "raw_products": f"{catalog_name}.{raw_schema}.{raw_tables['products']}",

    "enriched_customers": f"{catalog_name}.{enriched_schema}.{enriched_tables['customers']}",
    "enriched_products": f"{catalog_name}.{enriched_schema}.{enriched_tables['products']}",
    "fact_orders_enriched": f"{catalog_name}.{enriched_schema}.{enriched_tables['orders']}",

    "aggregate_profit": f"{catalog_name}.{aggregate_schema}.{aggregate_tables['profit_by_year_category_subcategory_customer']}"
}

print(target_tables)

In [0]:
print("Setup completed successfully.")